# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:
- Ángel González De La Vara
- Santiago Escobedo Vázquez   <br>

Url: https://github.com/santiblabla/MIAR_Algoritmos_Optimizacion/blob/main/03MIAR_Proyecto_Angel_Santiago/03MIAR_Proyecto_Angel_Santiago.ipynb<br>
Problema:
> 1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

- **Número de actores**: 10
- **Número de tomas**: 30
- **Actores/Tomas**: https://bit.ly/36D8IuK

1 indica que el actor participa en la toma. 0 en caso contrario.

(*) La respuesta es obligatoria                                   

En primer lugar importamos las librerias necesarias.

In [ ]:
import math
from sympy import bell
from io import StringIO
import numpy as np
import pandas as pd
from google.colab import drive
from pathlib import Path
import random

## Carga de datos del problema

Se carga el fichero de datos proporcionado por el enunciado del problema, directamente desde el Google Sheets. El fichero contiene los actores necesarios para cada una de las 30 tomas a realizar. Estos datos se cargan en una lista de listas.

In [ ]:
SHEET_ID = "1Ipn6IrbQP4ax8zOnivdBIw2lN0JISkJG4fXndYd27U0"

url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"
# La fila 1 (índice 1) contiene las cabeceras reales:
# Toma, 1, 2, ..., 10, ..., Total
df = pd.read_csv(url, header=1)

# Nos quedamos solo con las columnas de actores 1..10.
actor_cols = [str(i) for i in range(1, 11)]

# Conserva únicamente las filas que identifican una toma numérica,
# descartando la fila vacía y la fila final TOTAL.
df_tomas = df[pd.to_numeric(df["Toma"], errors="coerce").notna()].copy()

# Matriz: cada fila es una toma y cada columna un actor.
# Se fuerza el tipo entero para obtener exactamente 0 y 1.
tomas = df_tomas[actor_cols].astype(int).values.tolist()

print(tomas)

[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 1, 0, 1, 0, 0, 0], [1, 1, 0, 0, 0, 0, 1, 1, 0, 0], [0, 1, 0, 1, 0, 0, 0, 1, 0, 0], [1, 1, 0, 1, 1, 0, 0, 0, 0, 0], [1, 1, 0, 1, 1, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 1, 0, 0, 1, 0], [1, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 1, 1, 1, 0, 1, 0, 0, 0, 0], [1, 0, 0, 1, 1, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 0, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 1, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 1, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0, 0, 1], [1, 0, 1, 0, 1, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0], [1, 0, 0, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 1, 1, 0, 0, 0, 0], [1, 0, 0, 1, 0, 0, 0, 0, 0, 0]]


##(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>




**Respuesta**

Para responder esta pregunta se hace la hipótesis de que estamos hablando de días de grabación, es decir, no puede haber días vacíos en los que no se realiza ninguna toma.

Se busca encontrar el número de posibilidades de organizar las 30 tomas en días de grabación, teniendo en cuenta que el orden de las tomas en un mismo día no importa. Además, también se considera que el orden de los días tampoco importa ya que el coste para el estudio de grabación es el mismo y no se menciona en el enunciado ninguna restricción logística. Es decir, si imaginamos que tenemos que grabar 3 tomas A, B y C, desde un punto de vista de solución se considera que grabar las tomas A y B el día 1 y la toma C el día 2 es igual a grabar la toma C el día 1 y las tomas A y B y el día 2.

Sin tener en cuenta la restricción de un máximo de 6 tomas por día, el número de posibilidades se puede obtener mediante el **número de Bell**, que representa la cantidad de formas posibles de dividir un conjunto de n elementos en subconjuntos disjuntos y no vacíos. El número de Bell se calcula como:

$$
B_n = \sum_{k = 0}^{n-1} \binom{n-1}{k} B_k$$

Aplicándolo a nuestro problema con $n = 30$ (número de tomas), la solución es:


In [ ]:
# Calcular el número de Bell para n = 30
n_bell = bell(30)
print(n_bell)

846749014511809332450147


Por lo tanto, existen 846749014511809332450147 soluciones al problema sin restricciones.

**Nota extra:** si hubiésemos considerado que el orden de los días de grabación importan, por ejemplo, porque los actores solo tienen disponibilidad ciertos días o debido a otras razones logísticas, la solución se hubiese obtenido usando los números de Fubini (o números de Bell ordenados), que cuentan las formas de dividir un conjunto de n elementos en bloques no vacíos donde el orden de los bloques sí importa. Esta solución no se ha considerado debido a las hipótesis asumidas para la solución del problema, explicadas al principio de la respuesta a esta pregunta.

##¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones?

**Respuesta**

Si se considera la restricción de que se pueden grabar 6 tomas como máximo en un mismo día, el número de posibilidades se calcula mediante el **número de Bell restringido**, que representa las distintas formas de particionar un conjunto de n elementos bajo condiciones específicas, como limitar el tamaño de los subconjuntos. El número de Bell restringido se calcula como:

$$
B_{n, \le k} = \sum_{j = 1}^{min(n, k)} \binom{n-1}{j-1} B_{n-j, \le k}
$$

con $B_{0, \le k} = 1$ y siendo $n$ el número de tomas (30 en nuestro caso) y $k$ el número máximo de tomas por día (6 en nuestro caso).

Aplicandolo a nuestro problema obtenemos:

In [ ]:
def bell_restringido(n, k):
    # Inicializar la lista para almacenar los resultados previos
    B = [0] * (n + 1)
    B[0] = 1  # Caso base

    for i in range(1, n + 1):
        total = 0
        # Sumar las combinaciones válidas respetando el límite máximo 'k'
        for j in range(1, min(i, k) + 1):
            total += math.comb(i - 1, j - 1) * B[i - j]
        B[i] = total

    return B[n]

# Calcular para n = 30 y un máximo de 6 elementos por subconjunto
resultado = bell_restringido(30, 6)
print(f"Resultado: {resultado}")

Resultado: 726391948970868949621309


Por lo tanto existen 726391948970868949621309 soluciones al problema con la restricción de un máximo de 6 tomas por día.

Como curiosidad, nótese que el número de soluciones con restricciones es inferior al número de soluciones sin restricciones. Esto se debe a que las restricciones establecen como no validas algunas de las soluciones del espacio de soluciones sin restricciones.

##Modelo para el espacio de soluciones

##(*)¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)


**Respuesta**

La estructura de datos que mejor se adapta para representar una solución en este espacio es una **lista de listas**.

Se define una lista principal que representa el "calendario de rodaje". Dentro de esta, cada sublista representa un "día de grabación" (nótese que para el caso de nuestro problema habrá un mínimo 5 sublistas, que se corresponde con los 5 días resultantes de aplicar la restricción del máximo de 6 tomas por día, y un máximo de 30 sublistas, que se corresponde al caso de grabar una toma por día de forma que no queden días vacíos). Los elementos de cada sublista se corresponden con los números de las tomas asignadas a esa jornada (IDs de 1 a 30).

Para un ejemplo reducido de 3 tomas se tendría:

**Ejemplo de solución 1:**
- Día 1: Graba T1 y T2
- Día 2: Graba T3
- Solución: [[1, 2], [3]]

**Ejemplo de solución 2:**
- Día 1: Graba T1
- Día 2: Graba T3
- Día 3: Graba T2
- Solución: [[1], [3], [2]]

Esta estructura se adapta maravillosamente al problema por las siguientes razones:
- Representa de forma directa el modelo del problema, por lo que leer y entender una solución es inmediato.
- Facilita comprobar restricciones: el límite de 6 tomas por día se verifica con len(dia) <= 6, y que cada toma aparece una sola vez se comprueba recorriendo las sublistas.
- Facilita las operaciones necesarias para los algoritmos de resolución del problema.

#Según el modelo para el espacio de soluciones<br>
#(*)¿Cual es la función objetivo?



**Respuesta**

En primer lugar se formaliza matemáticamente la definición de la función objetivo.

Sea $T = \lbrace 1, 2, ..., n \rbrace$ el conjunto de IDs de las $n$ tomas ($n = 30$ en nuestro caso base).

Sea $A = \lbrace a_1, ..., a_m \rbrace$ el conjunto de $m$ actores de doblaje ($m = 10$ en nuestro caso base).

Para cada toma $t \in T$, sea $R_t \subseteq A$ el conjunto de actores que intervienen en la toma $t$.

Una solución (lista de listas como explicado anteriormente) se representa como:

$$
S = \lbrack D_1, D_2, ..., D_k \rbrack,
$$

donde:
- Cada $D_k$ es una lista (o conjunto) de tomas: $D_k \subseteq T$
- Los $D_k$ son disjuntos y su unión es $T$: $\bigcup_{k = 1}^{K} D_k = T$.
- Se cumple la restricción de capacidad $|D_k| \leq 6$ para todo $k$.

Para cada día $k$, el conjunto de actores que debe acudir es:

$$
A_k (S) = \bigcup_{t \in D_k} R_t
$$

Es decir, la unión de los actores de todas las tomas asignadas a ese día.

Considerando que todos los actores cuestan lo mismo por día y que este coste es igual a 1, **el coste total de una solución, o lo que es lo mismo, la función objetivo que se quiere optimizar, es**:

$$
C(S) = \sum_{k = 1}^{K} \left \vert \bigcup_{t \in D_k} R_t \right \vert ,
$$

que es el número total de pares "actor-día" en los que el actor tiene al menos una toma.

#(*)¿Es un problema de maximización o minimización?

**Respuesta**

Es un problema de minimización.

El objetivo es minimizar la función de coste $C(S)$ definida en el apartado anterior, lo que se corresponde con minimizar el número total de “actor-día” (pares actor, día en que el actor tiene al menos una toma).

#Diseña un algoritmo para resolver el problema por fuerza bruta

**Respuesta**

El algoritmo de fuerza bruta explora exhaustivamente todo el espacio de soluciones posibles, evaluando cada calendario de rodaje que cumple las restricciones y seleccionando aquel con el menor coste. La fuerza bruta garantiza encontrar la solución óptima, a costa de una complejidad temporal alta cuando aumenta el número de tomas.

Los pasos que sigue el algoritmo son los siguientes:

1. Generación de particiones acotadas: Se consideran las $n$ tomas (hasta 30 en nuestro caso) y se genera el conjunto de todas las particiones del conjunto $\lbrace 1,…,n\rbrace$ en bloques (días) no vacíos, imponiendo la restricción de que cada bloque tenga como máximo el número de tomas deseado (por ejemplo, 6 en nuestro caso). Cada partición representa un posible calendario de rodaje.

2. Representación de soluciones: Cada calendario se representa como una lista de listas. La lista principal contiene los días, y cada sublista contiene los IDs de tomas asignadas a ese día. Esta estructura de datos permite recorrer fácilmente las tomas de cada jornada.

3. Cálculo del coste por solución: Para cada calendario generado, se recorre cada día y se calcula el conjunto de actores que participan en alguna toma de ese día. El coste de la solución se obtiene sumando el número de actores distintos convocados en cada día.

4. Búsqueda de la mejor planificación: El algoritmo mantiene, durante el recorrido, la solución de menor coste encontrada hasta el momento. Tras evaluar todas las particiones posibles, la planificación almacenada es la solución óptima del problema, respetando el número máximo de tomas por día.

**NOTA**: Dado el coste computacional que requeriría evaluar 30 tomas con 10 actores, se evalúa un subconjunto de la matriz de actores en el que se utilizan 7 tomas con 3 actores, a fin encontrar una solución y comprobar el funcionamiento del algoritmo por fuerza bruta.

In [ ]:
def generar_particiones_acotadas(numero_tomas, maximo_tomas_dia):
    """
    Genera todas las particiones de {1, ..., numero_tomas} en bloques (días)
    de tamaño <= maximo_tomas_dia.
    Devuelve una lista de soluciones,
    donde cada solución es una lista de listas de tomas (IDs 1..numero_tomas).
    """
    soluciones = []

    def backtrack(siguiente_indice, bloques_actuales):
        # siguiente_indice recorre 0..numero_tomas-1 internamente
        if siguiente_indice == numero_tomas:
            # Canonizar bloques internamente (0-based)
            bloques_canonicos = [sorted(bloque) for bloque in bloques_actuales]
            bloques_canonicos.sort(key=lambda bloque: bloque[0])
            # Convertir a IDs 1..numero_tomas para devolver
            bloques_ids = [[id_toma + 1 for id_toma in bloque] for bloque in bloques_canonicos]
            soluciones.append(bloques_ids)
            return

        # Intentar añadir siguiente_indice a bloques existentes (si no se supera maximo_tomas_dia)
        for bloque in bloques_actuales:
            if len(bloque) < maximo_tomas_dia:
                bloque.append(siguiente_indice)
                backtrack(siguiente_indice + 1, bloques_actuales)
                bloque.pop()

        # Crear un bloque nuevo con siguiente_indice
        bloques_actuales.append([siguiente_indice])
        backtrack(siguiente_indice + 1, bloques_actuales)
        bloques_actuales.pop()

    backtrack(0, [])
    return soluciones


def coste_solucion(solucion_ids, tomas):
    """
    solucion_ids: lista de listas de tomas (IDs 1..numero_tomas).
    tomas: lista de listas con 0/1, tomas[i][j] = 1 si el actor j
           participa en la toma i (i = 0..numero_tomas-1).
    Devuelve la suma de actores distintos que vienen cada día.
    """
    coste_total = 0
    numero_actores = len(tomas[0])  # número de actores = longitud de cada fila

    for dia_ids in solucion_ids:
        actores_union = [0] * numero_actores  # vector binario de actores del día
        for id_toma_1based in dia_ids:
            indice_toma = id_toma_1based - 1   # convertir a índice 0-based
            fila_toma = tomas[indice_toma]     # actores de la toma
            for indice_actor in range(numero_actores):
                actores_union[indice_actor] = actores_union[indice_actor] or fila_toma[indice_actor]
        coste_total += sum(actores_union)

    return coste_total


def fuerza_bruta_calendario(tomas, numero_tomas=None, maximo_tomas_dia=6):
    """
    Explora todas las particiones de las tomas en bloques de tamaño <= maximo_tomas_dia
    y devuelve la planificación de coste mínimo y su coste.

    tomas: lista de listas de 0/1 (hasta 30 x numero_actores).
    numero_tomas: número de tomas a considerar (se usan tomas[0..numero_tomas-1]).
                  Si es None, se usan todas las tomas.
    maximo_tomas_dia: tamaño máximo de bloque (día), 6 en el problema original.
    """
    if numero_tomas is None:
        numero_tomas = len(tomas)

    tomas_consideradas = tomas[:numero_tomas]

    # Generamos soluciones en IDs 1..numero_tomas
    todas_particiones_ids = generar_particiones_acotadas(numero_tomas, maximo_tomas_dia)

    mejor_coste = None
    mejor_solucion_ids = None

    for solucion_ids in todas_particiones_ids:
        coste = coste_solucion(solucion_ids, tomas_consideradas)
        if mejor_coste is None or coste < mejor_coste:
            mejor_coste = coste
            mejor_solucion_ids = solucion_ids

    return mejor_coste, mejor_solucion_ids

In [ ]:
numero_tomas_demo = 7        # cuántas tomas queremos considerar (por ejemplo 7)
maximo_tomas_dia_demo = 3    # máximo de tomas por día (por ejemplo 3)

print("Número de tomas usadas en la fuerza bruta:", numero_tomas_demo)

# Generamos todas las particiones acotadas (soluciones en IDs 1..numero_tomas_demo)
particiones_demo_ids = generar_particiones_acotadas(numero_tomas_demo, maximo_tomas_dia_demo)
print("Número de particiones generadas:", len(particiones_demo_ids))

# Ejecutamos el algoritmo de fuerza bruta sobre 'tomas'
mejor_coste, mejor_solucion_ids = fuerza_bruta_calendario(
    tomas,
    numero_tomas=numero_tomas_demo,
    maximo_tomas_dia=maximo_tomas_dia_demo)

print("Mejor coste:", mejor_coste)
print("Mejor solución (IDs de tomas 1..{}):".format(numero_tomas_demo), mejor_solucion_ids)

Número de tomas usadas en la fuerza bruta: 7
Número de particiones generadas: 652
Mejor coste: 14
Mejor solución (IDs de tomas 1..7): [[1, 2, 6], [3, 4, 7], [5]]


#Calcula la complejidad del algoritmo por fuerza bruta

**Respuesta**

En el algoritmo de fuerza bruta, la complejidad temporal puede expresarse como:

$$
T(n, m, k) = O(B_{n, \leq k} \cdot n \cdot m),
$$

donde:

- $n$ es el número de tomas consideradas en el problema.
- $m$ es el número de actores.
- $k$ es el máximo número de tomas por día.
- $B_{n, \leq k}$ es el número de particiones del conjunto de $n$ tomas en bloques (días) no vacíos de tamaño $\leq k$, es decir, el número de Bell restringido explicado en uno de los ejercicios anteriores.

El término $B_{n, \leq k}$ refleja cuántas soluciones distintas (calendarios posibles) hay en el espacio de búsqueda; el algoritmo las genera todas y las evalúa una por una, por eso el tiempo es proporcional a ese número. el factor $n\cdot m$ procede del coste de evaluar cada solución: para cada calendario se recorre, en el peor caso, todas las tomas ($n$) y, por cada toma, todos los actores ($m$) para calcular la unión de actores por día y sumar la función objetivo. En conjunto, la complejidad es exponencial en $n$ (por el término $B_{n, \leq k}$) y lineal en el número de actores $m$, lo que hace que el algoritmo de fuerza bruta sea impracticable para $n$ grande.

Como complemento, se puede dar una cota explícita sustituyendo $B_{n, \leq k}$ por una expresión exponencial en función de $n$. El número de particiones de un conjunto de $n$ elementos (número de Bell clásico $B_n$) crece superexponencialmente, pudiéndose acotar de forma simple por $B_n = 2^{O(n \log n)}$. Para el caso restringido $B_{n, \leq k}$ con $k$ fijo, el crecimiento sigue siendo del mismo tipo pero con una constante que depende de $k$. Por tanto, se puede escribir:
$$
B_{n, \leq k} \leq 2^{c(k) \ n \log n}
$$

para alguna constante $c(k) > 0$, y sustituyendo en la expresión de complejidad:
$$
T(n,m,k) = O(2^{c(k) \ n \log n} \cdot n \cdot m)
$$

dejando claro que el algoritmo por fuerza bruta tiene complejidad superexponencial en $n$ y lineal en $m$, incluso cuando el límite de tomas por día $k$ es fijo.

#(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Para mejorar la complejidad temporal frente a un enfoque exhaustivo (como la fuerza bruta), se propone un algoritmo voraz. Este enfoque construye la solución paso a paso, tomando la mejor decisión local posible en cada momento sin dar marcha atrás. Esto permite encontrar soluciones para un gran número de tomas en un tiempo de ejecución drásticamente menor.

Para optimizar aún más los resultados, se ha introducido una heurística de ordenamiento previo. Los pasos que sigue el algoritmo son los siguientes:

1. Inicialización y Ordenamiento: Se genera una lista con todas las tomas pendientes (del 1 al 30). Inmediatamente, se ordenan de mayor a menor dificultad, calculando la dificultad en función del número total de actores que participan en cada toma.

2. Apertura de jornada: Se abre un "nuevo día de grabación" vacío.

3. Selección de la toma inicial: Se extrae la primera toma de la lista de pendientes y se añade al día. Gracias al ordenamiento del paso 1, nos aseguramos de que cada día comience siempre con la toma más compleja disponible.

4. Relleno voraz: Mientras el día no haya alcanzado su límite máximo (6 tomas) y sigan quedando tomas pendientes, el algoritmo evalúa todas las opciones restantes. Selecciona e incorpora aquella toma que genere el menor incremento de actores nuevos respecto a los que ya están convocados ese día. Una vez seleccionada, se elimina de las pendientes.

5. Cierre e iteración: Cuando el día alcanza su límite máximo de tomas, se cierra. Si aún quedan tomas en la lista de pendientes, se vuelve al paso 2 para abrir un nuevo día. Este ciclo se repite hasta que el rodaje esté completamente planificado.

In [ ]:

def coste_solucion_voraz(solucion_ids, matriz_tomas):
    """
    Calcula el coste total dado un calendario con IDs 1-based.
    """
    coste_total = 0
    for dia_ids in solucion_ids:
        if not dia_ids:
            continue
        # Convertir IDs 1-based a índices 0-based para acceder a la matriz
        indices_dia = [id_toma - 1 for id_toma in dia_ids]

        # Unión lógica (OR) de los vectores de actores para todas las tomas del día
        actores_dia = np.any(matriz_tomas[indices_dia], axis=0)
        coste_total += np.sum(actores_dia)

    return int(coste_total)


def algoritmo_voraz_calendario(tomas, maximo_tomas_dia=6):
    """
    Construye una planificación de rodaje utilizando una heurística voraz.
    Incluye una heurística de pre-ordenamiento: prioriza las tomas más complejas
    (más actores) al inicio.
    """
    matriz_tomas = np.array(tomas)
    num_tomas = len(matriz_tomas)

    # --- APLICACIÓN DE LA HEURÍSTICA DE ORDENAMIENTO ---
    # Ordenamos los índices de las tomas en función de la suma de sus actores (de mayor a menor).
    # np.sum(matriz_tomas[x]) devuelve la cantidad de actores en la toma 'x'.
    tomas_pendientes = sorted(
        range(num_tomas),
        key=lambda x: np.sum(matriz_tomas[x]),
        reverse=True
    )
    # ---------------------------------------------------

    calendario_0based = []

    while tomas_pendientes:
        dia_actual = []
        actores_del_dia = np.zeros(matriz_tomas.shape[1], dtype=int)

        # 1. Arrancamos el día con la primera toma disponible (que ahora será la más compleja restante)
        toma_inicial = tomas_pendientes.pop(0)
        dia_actual.append(toma_inicial)
        actores_del_dia = np.logical_or(actores_del_dia, matriz_tomas[toma_inicial])

        # 2. Rellenamos el día de forma voraz hasta el máximo permitido o hasta agotar tomas
        while len(dia_actual) < maximo_tomas_dia and tomas_pendientes:
            mejor_toma = -1
            menor_incremento = float('inf')

            for toma in tomas_pendientes:
                simulacion_actores = np.logical_or(actores_del_dia, matriz_tomas[toma])
                incremento_coste = np.sum(simulacion_actores) - np.sum(actores_del_dia)

                if incremento_coste < menor_incremento:
                    menor_incremento = incremento_coste
                    mejor_toma = toma

            dia_actual.append(mejor_toma)
            tomas_pendientes.remove(mejor_toma)
            actores_del_dia = np.logical_or(actores_del_dia, matriz_tomas[mejor_toma])

        calendario_0based.append(dia_actual)

    # Convertir formato de salida a IDs 1..numero_tomas
    mejor_solucion_ids = [[id_toma + 1 for id_toma in dia] for dia in calendario_0based]
    mejor_coste = coste_solucion_voraz(mejor_solucion_ids, matriz_tomas)

    return mejor_coste, mejor_solucion_ids

In [ ]:
# --- EJECUCIÓN DE DEMOSTRACIÓN ---
maximo_tomas_dia_demo = 6
mejor_coste_voraz, mejor_solucion_voraz_ids = algoritmo_voraz_calendario(tomas, maximo_tomas_dia_demo)
print("Mejor coste:", mejor_coste_voraz)
print("Mejor solución:", mejor_solucion_voraz_ids)

Mejor coste: 30
Mejor solución: [[1, 6, 7, 20, 22, 2], [11, 17, 19, 23, 4, 3], [12, 8, 9, 14, 18, 24], [10, 15, 29, 26, 13, 27], [25, 16, 28, 30, 5, 21]]


#(*)Calcula la complejidad del algoritmo

La complejidad del algoritmo voraz es $O(n^2 m)$, siendo:
- $n$: número de tomas.
- $m$: número de actores.

Ordenar las tomas por coste cuesta mirar todas las tomas y todos los actores ($O(nm)$) y luego ordenarlas ($O(n \log n)$). Al construir el calendario, en cada paso de rellenar un día se comparan todas las tomas que quedan con el día actual; al principio hay $n$, luego $n-1$, $n-2$, etc. Para cada comparación hay que revisar los $m$ actores, así que cada paso cuesta proporcional a "tomas pendientes multiplicado por $m$".

Sumando todas esas comparaciones:

$$
(n + (n-1) + ... + 1)\cdot m \approx \frac{n(n-1)}{2}\cdot m = O(n^2 m)
$$

El coste de ordenar y calcular el coste final son menores y no cambian el orden dominante, por lo que el coste del algoritmo voraz es cuadrático en $n$ y lineal en $m$.


#Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

In [ ]:
# Parámetros del problema
num_tomas = 30
num_actores = 10

# 1. Generar la matriz base con 0s y 1s
matriz = np.random.randint(0, 2, size=(num_tomas, num_actores))

# 2. Corrección vectorizada: Ninguna toma vacía (asignamos 1 actor al azar a las filas que sumen 0)
filas_vacias = np.where(matriz.sum(axis=1) == 0)[0]
matriz[filas_vacias, np.random.randint(0, num_actores, len(filas_vacias))] = 1

# 3. Corrección vectorizada: Ningún actor a cero (asignamos 1 toma al azar a las columnas que sumen 0)
cols_vacias = np.where(matriz.sum(axis=0) == 0)[0]
matriz[np.random.randint(0, num_tomas, len(cols_vacias)), cols_vacias] = 1

# Convertimos a una lista de listas para ser coherentes con la estructura definida al principio
tomas_aleatorias = matriz.tolist()

print(tomas_aleatorias)

[[1, 1, 1, 0, 0, 0, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 0, 1, 0], [1, 0, 0, 1, 1, 1, 0, 0, 1, 0], [1, 1, 1, 0, 1, 0, 1, 1, 1, 1], [0, 1, 1, 0, 1, 0, 1, 1, 0, 0], [0, 1, 0, 1, 0, 0, 0, 1, 0, 1], [0, 0, 1, 0, 1, 0, 0, 1, 0, 0], [1, 1, 0, 1, 1, 0, 1, 0, 1, 1], [1, 1, 0, 0, 1, 1, 0, 0, 0, 1], [0, 1, 1, 0, 1, 0, 0, 0, 1, 0], [1, 1, 0, 0, 0, 0, 0, 1, 1, 1], [1, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 1, 1, 0, 1, 0, 0, 1, 0, 0], [1, 0, 0, 0, 1, 0, 0, 0, 1, 0], [1, 0, 0, 1, 1, 1, 1, 1, 0, 1], [1, 1, 0, 1, 0, 1, 0, 1, 1, 0], [0, 1, 1, 0, 1, 1, 1, 0, 0, 1], [0, 0, 0, 1, 0, 0, 0, 0, 1, 1], [0, 0, 1, 0, 1, 1, 1, 0, 0, 0], [0, 1, 1, 1, 0, 1, 0, 1, 1, 1], [0, 0, 1, 0, 0, 0, 1, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 0, 0, 1, 0, 0, 1, 1], [0, 0, 0, 1, 1, 1, 0, 1, 1, 1], [1, 0, 0, 1, 0, 1, 1, 0, 1, 0], [0, 1, 0, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 1, 0, 1, 0, 0, 1], [0, 1, 0, 1, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 1, 1, 0, 0, 1]]


#Aplica el algoritmo al juego de datos generado

In [ ]:
# --- Ejecución del algoritmo voraz---
mejor_coste, mejor_solucion = algoritmo_voraz_calendario(tomas_aleatorias, maximo_tomas_dia=6)
print("Mejor coste:", mejor_coste)
print("Mejor solución:", mejor_solucion)

Mejor coste: 42
Mejor solución: [[2, 3, 26, 10, 12, 19], [4, 1, 5, 11, 13, 27], [8, 28, 14, 18, 29, 23], [15, 30, 25, 16, 9, 6], [20, 22, 24, 7, 17, 21]]


#Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

**Respuesta**

El enfoque de fuerza bruta es inviable cuando el número de tomas aumenta, debido a que el tiempo de computación crece de forma exponencial. Por otro lado, un algoritmo voraz (greedy) es eficiente, pero es susceptible a proporcionar una solución subóptima debido a que no se explora todo el espacio de soluciones.

Para mejorar la calidad de las soluciones con un sacrificio mínimo de velocidad, se propone:

- Enfoque estocástico (GRASP simplificado): En lugar de seleccionar siempre la opción con el menor incremento de coste, elegimos aleatoriamente entre las tres mejores opciones. Al ejecutar este proceso 100 veces, aumentamos drásticamente la probabilidad de encontrar una solución que se aproxime al óptimo global.



In [ ]:
import random
def coste_solucion_numpy(solucion_ids, matriz_tomas):
    """
    Calcula el coste total de un calendario (con IDs 1-based).
    """
    coste_total = 0
    for dia_ids in solucion_ids:
        if not dia_ids:
            continue
        # Convertir IDs 1-based a índices 0-based para acceder a la matriz
        indices_dia = [id_toma - 1 for id_toma in dia_ids]
        actores_dia = np.any(matriz_tomas[indices_dia], axis=0)
        coste_total += np.sum(actores_dia)

    return int(coste_total)


def algoritmo_grasp_calendario(tomas, maximo_tomas_dia=6, iteraciones=100, top_k=3):
    """
    Genera una planificación usando un enfoque estocástico (GRASP simplificado).
    Repite la construcción voraz aleatorizada 'iteraciones' veces y devuelve la mejor.
    Aplica la heurística de 'Toma más grande primero' para abrir los días.
    """
    matriz_tomas = np.array(tomas)
    num_tomas = len(matriz_tomas)

    mejor_coste_global = float('inf')
    mejor_solucion_global_ids = None

    # Bucle multi-arranque para explorar diferentes trayectorias
    for _ in range(iteraciones):
        tomas_pendientes = list(range(num_tomas))
        calendario_0based = []

        while tomas_pendientes:
            dia_actual = []
            actores_del_dia = np.zeros(matriz_tomas.shape[1], dtype=int)

            # 1. Novedad: Diversidad inicial usando "El más grande primero"
            candidatos_iniciales = []
            for t in tomas_pendientes:
                num_actores = np.sum(matriz_tomas[t])
                candidatos_iniciales.append((num_actores, t))

            # Ordenamos de MAYOR a MENOR número de actores
            candidatos_iniciales.sort(key=lambda x: x[0], reverse=True)

            # Restringimos las opciones a las top_k tomas más grandes
            mejores_iniciales = candidatos_iniciales[:min(top_k, len(candidatos_iniciales))]

            # Elegimos ALEATORIAMENTE entre las top_k más grandes para anclar el día
            _, toma_inicial = random.choice(mejores_iniciales)

            dia_actual.append(toma_inicial)
            tomas_pendientes.remove(toma_inicial)
            actores_del_dia = np.logical_or(actores_del_dia, matriz_tomas[toma_inicial])

            # 2. Rellenamos el día usando la heurística voraz (Menor incremento de coste)
            while len(dia_actual) < maximo_tomas_dia and tomas_pendientes:
                candidatos_relleno = []

                # Evaluamos el incremento de coste para todas las tomas restantes
                for toma in tomas_pendientes:
                    simulacion_actores = np.logical_or(actores_del_dia, matriz_tomas[toma])
                    incremento_coste = np.sum(simulacion_actores) - np.sum(actores_del_dia)
                    candidatos_relleno.append((incremento_coste, toma))

                # Ordenamos de MENOR a MAYOR incremento de coste
                candidatos_relleno.sort(key=lambda x: x[0])

                # Restringimos las opciones a las top_k mejores
                mejores_opciones = candidatos_relleno[:min(top_k, len(candidatos_relleno))]

                # Elegimos ALEATORIAMENTE entre las top_k mejores opciones
                _, mejor_toma = random.choice(mejores_opciones)

                dia_actual.append(mejor_toma)
                tomas_pendientes.remove(mejor_toma)
                actores_del_dia = np.logical_or(actores_del_dia, matriz_tomas[mejor_toma])

            calendario_0based.append(dia_actual)

        # Formatear la solución actual a IDs 1..numero_tomas
        solucion_actual_ids = [[id_toma + 1 for id_toma in dia] for dia in calendario_0based]
        coste_actual = coste_solucion_numpy(solucion_actual_ids, matriz_tomas)

        # Actualizar el récord global si esta iteración ha encontrado un calendario mejor
        if coste_actual < mejor_coste_global:
            mejor_coste_global = coste_actual
            mejor_solucion_global_ids = solucion_actual_ids

    return mejor_coste_global, mejor_solucion_global_ids

In [ ]:
# --- EJECUCIÓN DE DEMOSTRACIÓN ---

# Se configuran los parámetros
maximo_tomas_dia_demo = 6
numero_de_iteraciones = 5000
tamaño_top_k = 3

print(f"--- EJECUCIÓN DEL ALGORITMO GRASP ({numero_de_iteraciones} Iteraciones, Top-{tamaño_top_k}) ---")
mejor_coste_grasp, mejor_solucion_grasp_ids = algoritmo_grasp_calendario(
    tomas,
    maximo_tomas_dia=maximo_tomas_dia_demo,
    iteraciones=numero_de_iteraciones,
    top_k=tamaño_top_k
)

print("Mejor coste encontrado:", mejor_coste_grasp)
print("Mejor solución (IDs de tomas 1..{}):".format(len(tomas)), mejor_solucion_grasp_ids)

--- EJECUCIÓN DEL ALGORITMO GRASP (5000 Iteraciones, Top-3) ---
Mejor coste encontrado: 28
Mejor solución (IDs de tomas 1..30): [[11, 19, 17, 1, 5, 6], [12, 14, 18, 22, 8, 24], [10, 15, 3, 29, 4, 21], [25, 16, 30, 28, 7, 9], [2, 27, 13, 20, 26, 23]]
